# Marketing A/B Test Analysis
**Did the ad campaign actually work, and is it worth running at scale?**

Dataset: [Marketing A/B Testing](https://www.kaggle.com/datasets/faviovaz/marketing-ab-testing) (Kaggle) — 588,101 users, randomly assigned to see either an ad (`ad`, treatment) or a public service announcement (`psa`, control). Target: `converted` (did the user purchase).

This notebook:
1. Loads and validates the data
2. Tests whether the ad group's conversion rate is significantly higher than control
3. Segments the effect to see where it holds vs. breaks down
4. Builds a break-even cost model across 9 business-assumption scenarios
5. Ends with a conditional rollout recommendation


In [1]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
df = pd.read_csv('../data/marketing_AB.csv')
df.head()

,user_id,test_group,converted,total_ads,most_ads_day,most_ads_hour
0,1069124,ad,0,130,Monday,20
1,1119715,ad,0,93,Tuesday,22
2,1144181,ad,0,21,Tuesday,18
3,1435133,ad,0,355,Tuesday,10
4,1015700,ad,0,276,Friday,14


## 1. Load & validate

In [2]:
print(f"Rows: {len(df):,}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate user_ids: {df['user_id'].duplicated().sum()}")
print()
print(df['test_group'].value_counts())

Rows: 588,101
Missing values: 0
Duplicate rows: 0
Duplicate user_ids: 0

test_group
ad     564577
psa     23524
Name: count, dtype: int64


No missing values, no duplicates. Group sizes are heavily imbalanced (564,577 ad vs. 23,524 psa) — this is realistic for a real ad platform rollout, where most traffic goes to the live campaign and only a small holdout sees the control. The z-test below accounts for this imbalance directly (it doesn't require equal group sizes).

## 2. Baseline conversion rates & significance test

In [3]:
conv = df.groupby('test_group')['converted'].agg(conversions='sum', users='count', rate='mean')
conv

,conversions,users,rate
test_group,,,
ad,14423,564577,0.0255
psa,420,23524,0.0179


In [4]:
ad = df.loc[df.test_group == 'ad', 'converted']
psa = df.loc[df.test_group == 'psa', 'converted']

count = [ad.sum(), psa.sum()]
nobs = [len(ad), len(psa)]

z_stat, p_value = proportions_ztest(count, nobs, alternative='two-sided')
ci_low, ci_high = confint_proportions_2indep(count[0], nobs[0], count[1], nobs[1], method='wald')

ad_rate, psa_rate = count[0]/nobs[0], count[1]/nobs[1]
diff_pp = (ad_rate - psa_rate) * 100
relative_lift = (ad_rate - psa_rate) / psa_rate * 100

print(f"Ad group conversion:  {ad_rate*100:.4f}%")
print(f"PSA group conversion: {psa_rate*100:.4f}%")
print(f"Absolute lift:        {diff_pp:.2f} percentage points")
print(f"Relative lift:        {relative_lift:.1f}%")
print(f"z-statistic:          {z_stat:.3f}")
print(f"p-value:              {p_value:.2e}")
print(f"95% CI on lift:       [{ci_low*100:.2f}, {ci_high*100:.2f}] pp")

Ad group conversion:  2.5547%
PSA group conversion: 1.7854%
Absolute lift:        0.77 percentage points
Relative lift:        43.1%
z-statistic:          7.370
p-value:              1.71e-13
95% CI on lift:       [0.60, 0.94] pp


**Verdict:** the lift is statistically significant — p < 0.001, and the 95% confidence interval (0.60–0.94 pp) doesn't cross zero. The ad group converts at roughly a **43% higher rate** (relative) than the control group. This isn't noise.

## 3. Segmentation — where does the effect actually hold?

In [5]:
ad_only = df[df.test_group == 'ad'].copy()
bins = [0, 10, 25, 50, 100, 200, ad_only['total_ads'].max()]
ad_only['ad_bucket'] = pd.cut(ad_only['total_ads'], bins=bins)
ad_only.groupby('ad_bucket', observed=True)['converted'].agg(conversion_rate='mean', users='count')

,conversion_rate,users
ad_bucket,,
"(0, 10]",0.0033,249499
"(10, 25]",0.0102,163135
"(25, 50]",0.0354,85740
"(50, 100]",0.1163,44149
"(100, 200]",0.1768,16360
"(200, 2065]",0.1558,5694


Conversion climbs steadily as users see more ads, up through the 100–200 range — then **drops** in the 200+ bucket. That's a diminishing (and eventually negative) returns signal: blasting a small number of users with an extreme number of ads isn't the efficient way to spend budget.

In [6]:
day = (df.groupby(['most_ads_day', 'test_group'], observed=True)['converted']
         .mean().unstack('test_group')
         .assign(lift_pp=lambda x: (x['ad'] - x['psa']) * 100)
         .sort_values('lift_pp', ascending=False))
day

test_group,ad,psa,lift_pp
most_ads_day,,,
Tuesday,0.0304,0.0144,1.5992
Monday,0.0332,0.0226,1.0683
Wednesday,0.0254,0.0158,0.9597
Saturday,0.0213,0.0140,0.7311
Friday,0.0225,0.0163,0.6162
Sunday,0.0246,0.0206,0.4025
Thursday,0.0216,0.0202,0.1407


The lift isn't evenly spread across the week. **Tuesday and Monday show the strongest lift** (1.60pp and 1.07pp); **Thursday is nearly flat** (0.14pp). A rollout that concentrates spend early in the week would capture more of the effect per dollar than a flat, every-day campaign.

## 4. Break-even cost model — is this worth running?

Statistical significance tells us the effect is real. It doesn't tell us whether the campaign pays for itself — that depends on two numbers the dataset doesn't give us: **average order value (AOV)** and **cost per ad impression**. Both are assumptions, stated explicitly, tested across a 3x3 grid (9 scenarios) rather than picked once and hidden.

In [7]:
import itertools

AOV_SCENARIOS = [20, 35, 50]                  # average order value, $
COST_PER_AD_SCENARIOS = [0.02, 0.05, 0.10]    # cost per ad impression, $

lift_rate = ad_rate - psa_rate
n_reach = len(ad)
avg_ads_per_user = ad_only['total_ads'].mean()

rows = []
for aov, cost_per_ad in itertools.product(AOV_SCENARIOS, COST_PER_AD_SCENARIOS):
    incr_conversions = lift_rate * n_reach
    incr_revenue = incr_conversions * aov
    campaign_cost = n_reach * avg_ads_per_user * cost_per_ad
    net_value = incr_revenue - campaign_cost
    breakeven_cost_per_ad = incr_revenue / (n_reach * avg_ads_per_user)
    rows.append({
        'aov_usd': aov, 'cost_per_ad_usd': cost_per_ad,
        'incremental_conversions': round(incr_conversions),
        'incremental_revenue_usd': round(incr_revenue, 2),
        'campaign_cost_usd': round(campaign_cost, 2),
        'net_value_usd': round(net_value, 2),
        'breakeven_cost_per_ad_usd': round(breakeven_cost_per_ad, 4),
        'recommendation': 'ship' if net_value > 0 else 'hold',
    })

scenarios = pd.DataFrame(rows)
scenarios

,aov_usd,cost_per_ad_usd,incremental_conversions,incremental_revenue_usd,campaign_cost_usd,net_value_usd,breakeven_cost_per_ad_usd,recommendation
0,20,0.0200,4343,"86,859.6400","280,294.0200","-193,434.3800",0.0062,hold
1,20,0.0500,4343,"86,859.6400","700,735.0500","-613,875.4100",0.0062,hold
2,20,0.1000,4343,"86,859.6400","1,401,470.1000","-1,314,610.4600",0.0062,hold
3,35,0.0200,4343,"152,004.3800","280,294.0200","-128,289.6400",0.0108,hold
4,35,0.0500,4343,"152,004.3800","700,735.0500","-548,730.6700",0.0108,hold
5,35,0.1000,4343,"152,004.3800","1,401,470.1000","-1,249,465.7200",0.0108,hold
6,50,0.0200,4343,"217,149.1100","280,294.0200","-63,144.9100",0.0155,hold
7,50,0.0500,4343,"217,149.1100","700,735.0500","-483,585.9400",0.0155,hold
8,50,0.1000,4343,"217,149.1100","1,401,470.1000","-1,184,320.9900",0.0155,hold


**Every one of the 9 scenarios nets negative at full-population, flat-rate rollout** — average exposure (~25 ads/user) is expensive relative to a ~0.77pp lift once basic ad costs are applied. The break-even cost per ad ranges from **$0.0062 to $0.0155** depending on AOV — well under the $0.02–$0.10 range tested. That's a real, useful finding: a blanket rollout to everyone, at typical ad costs, likely loses money.

## 5. Recommendation

**Conditional rollout, not a blanket one.** The lift is real and significant, but a flat, everyone-gets-25-ads campaign doesn't clear break-even at any tested cost/AOV combination. Two findings from the segmentation point to a cheaper path to the same lift:

1. **Cap exposure per user** — conversion gains taper (and reverse) past ~200 ads/user. Capping exposure around the 50–100 ad range likely keeps most of the lift at a fraction of the cost.
2. **Weight spend toward Monday/Tuesday** — those days show 3–4x the lift of the weakest days (Thursday, Sunday). Shifting budget toward the strongest days improves revenue-per-dollar-spent without touching total reach.

**Next step before a full rollout:** run a smaller, targeted pilot (capped exposure, Monday/Tuesday-weighted) and re-measure the break-even math on that population before committing full budget — rather than shipping or killing the campaign on the uncapped, unsegmented numbers alone.

*Limitations: AOV and cost-per-ad are modeling assumptions, not observed data — real values from the business would replace these. The dataset also doesn't include post-campaign retention or repeat-purchase behavior, which a full rollout decision should account for.*